This notebook produces animations of the different fields for comparing geostrophy and cyclogeostrophy using L3-SWOT data:

The different reconstruction methods:

- Geostrophic (included in SWOT L3 product, from Tranchant et al.),
- Cyclogeostrophic:
  - Minimization-based,
  - Gradient-wind,
  - Fixed-point.

For each reconstruction method:

- Velocity ($u$, $v$, and $\|\mathbf{u}\|$),
- Velocity anomaly ($u'$, $v'$, and $\|\mathbf{u}'\|$),
- Relative vorticity,
- Kinetic energy,
- Eddy kinetic energy.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import cartopy.crs as ccrs
import cmocean.cm as cmo
from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [ ]:
swot_ds = xr.open_zarr("./data/SWOT_L3_Expert_v3_0_pass003_pass016_cg.zarr")

In [ ]:
min_longitude = swot_ds.attrs["min_longitude"]
max_longitude = swot_ds.attrs["max_longitude"]
min_latitude = swot_ds.attrs["min_latitude"]
max_latitude = swot_ds.attrs["max_latitude"]

In [ ]:
def do_animation(
    ds: xr.Dataset,
    varnames: list[str],
    method_names: list[str],
    cmap,
    label: str,
    ncol_wrap: int = 2,
    units: str = None,
    interval: int = 500,
    save_dir: str = None,
):
    """
    Create an animation of several (related) fields, optionally saving frames as PNG images.

    Parameters
    ----------
    ds : xr.Dataset
        xarray Dataset to animate.
    varnames : list[str]
        List of variable names to plot.
    method_names : list[str]
        List of method names corresponding to the variable names, used for labeling subplots.
    cmap : colormap
        Colormap to use for the plots.
    label : str
        Label for the colorbar.
    units : str, optional
        Units for the colorbar label. If provided, displayed as "{label} ({units})".
    interval : int, optional
        Interval between frames in milliseconds. Default is 500.
    save_dir : str, optional
        Directory to save frames as PNG images.

    Returns
    -------
    IPython.display.HTML
        HTML5 video of the animation.
    """
    from_cycle = ds.cycle_num.min().values
    to_cycle = ds.cycle_num.max().values

    da = xr.concat(
        [ds[varname].stack(dim={"y": ("pass_num", "num_lines")}) for varname in varnames], 
        dim=xr.Variable("reconstruction", method_names), coords="different", compat="equals"
    )

    g = da.sel(cycle_num=from_cycle).plot.pcolormesh(
        subplot_kws=dict(projection=ccrs.PlateCarree()),
        transform=ccrs.PlateCarree(),
        x="longitude",
        y="latitude",
        col="reconstruction",
        col_wrap=ncol_wrap,
        cmap=cmap,
        robust=True,
        cbar_kwargs={
            "label": f"{label} ({units})" if units else label
        }
    )

    title = g.fig.suptitle(f"Cylcle {from_cycle}", y=1.05)

    g.set_titles("{value}")

    quadmesh_list = []
    for ax in g.axs.flat:
        ax.coastlines()
        ax.set_extent([min_longitude, max_longitude, min_latitude, max_latitude], crs=ccrs.PlateCarree())
        for child in ax.get_children():
            if isinstance(child, plt.matplotlib.collections.QuadMesh):
                quadmesh_list.append(child)

    def draw_frame(i):
        for j, method_name in enumerate(method_names):
            quadmesh = quadmesh_list[j]
            arr = da.sel(cycle_num=i, reconstruction=method_name).values
            arr = np.squeeze(arr)
            try:
                quadmesh.set_array(arr)
            except Exception:
                quadmesh.set_array(np.full(arr.shape, np.nan))

        title.set_text(f"Cycle {i}")

        return *quadmesh_list, title

    frames = np.arange(from_cycle, to_cycle + 1)

    # Save frames as PNG if requested
    if save_dir:
        save_dir = os.path.join("plots/L3", save_dir)
        os.makedirs(save_dir, exist_ok=True)
        n_digits = len(str(frames[-1]))
        for idx, i in enumerate(frames):
            draw_frame(i)
            fname = f"{str(idx).zfill(n_digits)}.png"
            path = os.path.join(save_dir, fname)
            g.fig.savefig(path, dpi=300, bbox_inches="tight")
        plt.close(g.fig)

    anim = animation.FuncAnimation(
        g.fig, draw_frame, init_func=lambda: draw_frame(from_cycle), frames=frames, interval=interval, blit=True
    )

    video = anim.to_html5_video()

    plt.close(g.fig)

    return HTML(video)

## Velocity

In [ ]:
plt.ioff()
do_animation(
    swot_ds,
    varnames=["ugos_filtered", "ucg_mb", "ucg_gw", "ucg_fp"],
    method_names=["Geostrophy", "Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="$u$", units="m s$^{-1}$"
)

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["vgos_filtered", "vcg_mb", "vcg_gw", "vcg_fp"],
    method_names=["Geostrophy", "Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="$v$", units="m s$^{-1}$"
)

In [ ]:
swot_ds["uvgos_filtered"] = np.sqrt(swot_ds.ugos_filtered**2 + swot_ds.vgos_filtered**2)
swot_ds["uvcg_mb"] = np.sqrt(swot_ds.ucg_mb**2 + swot_ds.vcg_mb**2)
swot_ds["uvcg_gw"] = np.sqrt(swot_ds.ucg_gw**2 + swot_ds.vcg_gw**2)
swot_ds["uvcg_fp"] = np.sqrt(swot_ds.ucg_fp**2 + swot_ds.vcg_fp**2)

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["uvgos_filtered", "uvcg_mb", "uvcg_gw", "uvcg_fp"],
    method_names=["Geostrophy", "Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.speed, label="$\|\mathbf{u}\|$", units="m s$^{-1}$",
    save_dir="speed"
)

In [ ]:
swot_ds["uvc_mb"] = swot_ds.uvcg_mb - swot_ds.uvgos_filtered
swot_ds["uvc_gw"] = swot_ds.uvcg_gw - swot_ds.uvgos_filtered
swot_ds["uvc_fp"] = swot_ds.uvcg_fp - swot_ds.uvgos_filtered

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["uvc_mb", "uvc_gw", "uvc_fp"],
    method_names=["Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="$\|\mathbf{u}_c\| = \|\mathbf{u}_{cg}\| - \|\mathbf{u}_g\|$", units="m s$^{-1}$",
    ncol_wrap=3,
    save_dir="cyclostrophic_correction"
)

## Relative vorticity

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["nrv_g", "nrv_cg_mb", "nrv_cg_gw", "nrv_cg_fp"],
    method_names=["Geostrophy", "Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.curl, label="$\zeta/f$", units=None,
    save_dir="normalized_relative_vorticity"
)

In [ ]:
swot_ds["nrv_diff_mb"] = swot_ds.nrv_cg_mb - swot_ds.nrv_g
swot_ds["nrv_diff_gw"] = swot_ds.nrv_cg_gw - swot_ds.nrv_g
swot_ds["nrv_diff_fp"] = swot_ds.nrv_cg_fp - swot_ds.nrv_g

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["nrv_diff_mb", "nrv_diff_gw", "nrv_diff_fp"],
    method_names=["Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="$(\zeta_{cg} - \zeta_g)/f$", units=None,
    ncol_wrap=3,
    save_dir="normalized_relative_vorticity_diff"
)

## Okubo-Weiss parameter

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["ow_g", "ow_cg_mb", "ow_cg_gw", "ow_cg_fp"],
    method_names=["Geostrophy", "Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="Okubo-Weiss parameter", units=None,
    save_dir="okubo-weiss"
)

In [ ]:
swot_ds["ow_diff_mb"] = swot_ds.ow_cg_mb - swot_ds.ow_g
swot_ds["ow_diff_gw"] = swot_ds.ow_cg_gw - swot_ds.ow_g
swot_ds["ow_diff_fp"] = swot_ds.ow_cg_fp - swot_ds.ow_g

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["ow_diff_mb", "ow_diff_gw", "ow_diff_fp"],
    method_names=["Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="Okubo-Weiss difference", units=None,
    ncol_wrap=3,
    save_dir="okubo-weiss_diff"
)

## Kinetic energy

In [ ]:
plt.ioff()
do_animation(
    swot_ds, 
    varnames=["ke_g", "ke_cg_mb", "ke_cg_gw", "ke_cg_fp"],
    method_names=["Geostrophy", "Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.amp, label="KE = $(u^2 + v^2)/2$", units="m$^2$ s$^{-2}$",
    save_dir="kinetic_energy"
)

In [ ]:
swot_ds["ke_diff_mb"] = swot_ds.ke_cg_mb - swot_ds.ke_g
swot_ds["ke_diff_gw"] = swot_ds.ke_cg_gw - swot_ds.ke_g
swot_ds["ke_diff_fp"] = swot_ds.ke_cg_fp - swot_ds.ke_g

In [ ]:
plt.ioff()
do_animation(
    swot_ds,
    varnames=["ke_diff_mb", "ke_diff_gw", "ke_diff_fp"],
    method_names=["Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="KE$_{cg}$ - KE$_g$", units="m$^2$ s$^{-2}$",
    ncol_wrap=3,
    save_dir="kinetic_energy_diff"
)

## Eddy kinetic energy

In [ ]:
plt.ioff()
do_animation(
    swot_ds,
    varnames=["eke_g", "eke_cg_mb", "eke_cg_gw", "eke_cg_fp"],
    method_names=["Geostrophy", "Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.amp, label="EKE = $(u'^2 + v'^2)/2$", units="m$^2$ s$^{-2}$",
    save_dir="eddy_kinetic_energy"
)

In [ ]:
swot_ds["eke_diff_mb"] = swot_ds.eke_cg_mb - swot_ds.eke_g
swot_ds["eke_diff_gw"] = swot_ds.eke_cg_gw - swot_ds.eke_g
swot_ds["eke_diff_fp"] = swot_ds.eke_cg_fp - swot_ds.eke_g

In [ ]:
plt.ioff()
do_animation(
    swot_ds,
    varnames=["eke_diff_mb", "eke_diff_gw", "eke_diff_fp"],
    method_names=["Minimization-based", "Gradient-wind", "Fixed-point"],
    cmap=cmo.balance, label="EKE$_{cg}$ - EKE$_g$", units="m$^2$ s$^{-2}$",
    ncol_wrap=3,
    save_dir="eddy_kinetic_energy_diff"
)